In [4]:
!pip install -q huggingface_hub pandas


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from huggingface_hub import InferenceClient
import pandas as pd
import time

HF_TOKEN = "..."

QUESTION = """
Xin luật sư cho tôi hỏi:
mua lại công ty đang hoạt động thì người nhận chuyển nhượng cần kiểm tra rủi ro thuế gì?
"""

MODELS = [
{
"name": "CMC-AI-Legal-32B",
"id": "CMC-OPENAI/CMC-AI-Legal-32B"
},
{
"name": "ChatLaw-13B",
"id": "pandalla/ChatLaw-13B"
},
{
"name": "LawGPT-7B",
"id": "lawgpt/LawGPT-7B-beta1.0"
},
{
"name": "Lawyer-LLaMA-13B",
"id": "AndrewZhe/lawyer-llama-13b"
},
{
"name": "SaulLM-7B",
"id": "Equall/Saul-7B-Instruct-v1"
},
{
"name": "KL3M",
"id": "alea-institute/kl3m-3b"
}
]

client = InferenceClient(api_key=HF_TOKEN)

In [ ]:
from huggingface_hub import InferenceClient

HF_TOKEN = "..."

client = InferenceClient(
    api_key=HF_TOKEN
)

print("Connected!")

Connected!


In [8]:
results = []

for model in MODELS:

    print("=" * 80)
    print("MODEL:", model["name"])

    answer = ""
    status = ""

    try:

        try:
            response = client.chat_completion(
                model=model["id"],
                messages=[
                    {
                        "role": "user",
                        "content": QUESTION
                    }
                ],
                max_tokens=512
            )

            answer = response.choices[0].message.content
            status = "SUCCESS"

        except Exception:

            answer = client.text_generation(
                QUESTION,
                model=model["id"],
                max_new_tokens=512
            )

            status = "SUCCESS"

    except Exception as e:

        status = f"FAILED: {str(e)}"
        answer = ""

    results.append({
        "Model": model["name"],
        "Model_ID": model["id"],
        "Question": QUESTION.strip(),
        "Answer": answer,
        "Status": status
    })

df = pd.DataFrame(results)

df.to_csv(
    "results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved results.csv")
display(df)

MODEL: CMC-AI-Legal-32B
MODEL: ChatLaw-13B
MODEL: LawGPT-7B
MODEL: Lawyer-LLaMA-13B
MODEL: SaulLM-7B
MODEL: KL3M
Saved results.csv


,Model,Model_ID,Question,Answer,Status
0,CMC-AI-Legal-32B,CMC-OPENAI/CMC-AI-Legal-32B,Xin luật sư cho tôi hỏi:\nmua lại công ty đang...,,FAILED:
1,ChatLaw-13B,pandalla/ChatLaw-13B,Xin luật sư cho tôi hỏi:\nmua lại công ty đang...,,FAILED:
2,LawGPT-7B,lawgpt/LawGPT-7B-beta1.0,Xin luật sư cho tôi hỏi:\nmua lại công ty đang...,,FAILED: 401 Client Error. (Request ID: Root=1-...
3,Lawyer-LLaMA-13B,AndrewZhe/lawyer-llama-13b,Xin luật sư cho tôi hỏi:\nmua lại công ty đang...,,FAILED: 401 Client Error. (Request ID: Root=1-...
4,SaulLM-7B,Equall/Saul-7B-Instruct-v1,Xin luật sư cho tôi hỏi:\nmua lại công ty đang...,,FAILED: Model Equall/Saul-7B-Instruct-v1 is no...
5,KL3M,alea-institute/kl3m-3b,Xin luật sư cho tôi hỏi:\nmua lại công ty đang...,,FAILED: 401 Client Error. (Request ID: Root=1-...
